# Darcy equation: exercise 5

Let $\Omega=(0,1)^3$ with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Given 
$k$ the matrix permeability, we want to solve the following problem: find $({q}, p)$ such that
$$
\left\{
\begin{array}{ll}
\begin{array}{l} 
k^{-1} {q} + \nabla p = 0\\
\nabla \cdot {q} = 0
\end{array}
&\text{in } \Omega
\end{array}
\right.
$$
with boundary conditions:
$$ p = 0 \text{ on } \partial_{top} \Omega \qquad p = 1 \text{ on } \partial_{bottom} \Omega \qquad \nu \cdot q = 0 \text{ on } \partial_{left} \Omega \cup \partial_{right} \Omega \cup \partial_{front} \Omega \cup \partial_{back} \Omega$$
The matrix permeability is defined in the following way
$$
k(x, y, z) = 
\left\{
\begin{array}{ll}
k_1 & 0.2 < z < 0.4 \text{ and } x < 0.5 \text{ and } y < 0.5\\
k_2 & 0.6 < z < 0.8 \text{ and } x > 0.5\\
1 & \text{otherwise}
\end{array}
\right.
$$
with, for example, $k_1 = k_2 = 10^{-2}$.

This is the guided ("fill in the code") version of `ex5.ipynb` -- work through the cells in order, replacing each `# TODO` with your own implementation. Compare against `ex5.ipynb` once you're done, or if you get stuck.

First we import some of the standard modules, like `numpy` and `scipy.sparse`. Since PyGeoN is based on [PorePy](https://github.com/pmgbergen/porepy) we import both modules.

In [ ]:
import numpy as np
import scipy.sparse as sps

import porepy as pp
import pygeon as pg

We create now the grid, to facilitate the imposition of $k$ we consider a structured grid from PorePy and then convert it into a PyGeoN grid.

In [ ]:
# TODO: create a 3d STRUCTURED grid with pg.unit_grid(dim, 1 / N, as_mdg=False, structured=True)
# (try N = 10 to start), then call sd.compute_geometry()


Let us declare the finite element spaces that we are going to use

In [ ]:
# TODO: declare the RT0 (for q) and PwConstants/P0 (for p) discretization
# objects under a key of your choice, e.g. key = "flow"
#
# TODO: build the degrees-of-freedom array
# dofs = np.array([rt0.ndof(sd), p0.ndof(sd)])


With the following code we set the data, in particular the permeability tensor and the boundary conditions. Since we need to identify each side of $\partial \Omega$ we need few steps.

In [ ]:
# TODO: build a heterogeneous (inverse) permeability array over the cells, using
# sd.cell_centers for x, y, z: 1/k1 in the first box, 1/k2 in the second, 1
# (i.e. k0 = 1) everywhere else -- see the piecewise definition above.
# Pack it with pp.SecondOrderTensor and pp.initialize_data (see pg.SECOND_ORDER_TENSOR)
#
# TODO: identify the six sides of the domain from sd.face_centers
# (left/right, back/front, bottom/top)
#
# TODO: impose p = 1 on bottom and p = 0 on top as a NATURAL boundary condition for
# RT0 (use rt0.assemble_nat_bc with a function returning the pressure value on the
# boundary), and nu.q = 0 (essential) on the other four sides


Once the data are assigned to the grid, we construct the matrices. In particular, the linear system associated with the equation is given as
$$
\left(
\begin{array}{cc} 
A & -B^\top\\
B & 0
\end{array}
\right)
\left(
\begin{array}{c} 
q\\ 
p
\end{array}
\right)
=\left(
\begin{array}{c} 
p_{\partial}\\ 
0
\end{array}
\right)
$$<br>
where $p_{\partial}$ is the vector associated to the pressure boundary conditions.

In [ ]:
# TODO: assemble the local matrices -- the RT0 mass matrix A (with data), the P0 mass
# matrix, and the divergence matrix B = mass_p0 @ rt0.assemble_diff_matrix(sd)
#
# TODO: assemble the saddle-point matrix spp with scipy.sparse.block_array
# ([[A, -B.T], [B, None]], format="csc")
#
# TODO: assemble the right-hand side rhs (length dofs.sum()), adding the boundary
# term to the q-block (the first dofs[0] entries)


We solve the linear system and extract the two solutions $q$ and $p$.

In [ ]:
# TODO: build a pg.LinearSystem from spp and rhs, flag the essential boundary dofs
# with ls.flag_ess_bc(bc_ess, ...), then solve()
#
# TODO: split the solution vector into q and p, e.g. with
# idx = np.cumsum(dofs[:-1]); q, p = np.split(x, idx)


Since the computed $q$ is one value per facet of the grid, for visualization purposes we project the flux in each cell center as vector. We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [ ]:
# TODO: project q to cell centers with rt0.eval_at_cell_centers(sd), and evaluate p
# at cell centers with p0.eval_at_cell_centers(sd)
#
# TODO: export cell_p, cell_q and the permeability array with
# pp.Exporter(sd, "sol", folder_name="ex5").write_vtu(...)


In [ ]:
# Consistency check -- once your implementation is correct, this should pass
assert np.isclose(np.linalg.norm(cell_p), 44.12881151508367)
assert np.isclose(np.linalg.norm(cell_q), 54.76964472280665)